In [15]:
"""
Q1 — Dataset Preparation (Tiny ImageNet → 100-class subset with AlexNet-style preprocessing)


Data source: Tiny ImageNet (Stanford CS231n) — tiny-imagenet-200.zip
What this notebook does:
  1) Unzips Tiny ImageNet once.
  2) Selects exactly 100 classes and takes exactly 500 images per class.
  3) Splits each class into Train=300, Val=100, Test=100 (→ 30k / 10k / 10k).
  4) Writes CSV manifests (train.csv, val.csv, test.csv) and meta.json summary.
  5) Provides AlexNet Section-2 preprocessing (Resize 256 → 224 crop → flips → normalize).
"""

'\nQ1 — Dataset Preparation (Tiny ImageNet → 100-class subset with AlexNet-style preprocessing)\n\n\nData source: Tiny ImageNet (Stanford CS231n) — tiny-imagenet-200.zip\nWhat this notebook does:\n  1) Unzips Tiny ImageNet once.\n  2) Selects exactly 100 classes and takes exactly 500 images per class.\n  3) Splits each class into Train=300, Val=100, Test=100 (→ 30k / 10k / 10k).\n  4) Writes CSV manifests (train.csv, val.csv, test.csv) and meta.json summary.\n  5) Provides AlexNet Section-2 preprocessing (Resize 256 → 224 crop → flips → normalize).\n'

In [16]:
from pathlib import Path

# point to my downloaded zip
ZIP_PATH = Path("./tiny-imagenet-200.zip")

# where to extract Tiny ImageNet and where to write the final subset
DATA_DIR = Path("./data").resolve()
OUT_DIR  = Path("./tiny_100_alexnet").resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR, OUT_DIR


(WindowsPath('C:/Users/veda2/data'),
 WindowsPath('C:/Users/veda2/tiny_100_alexnet'))

In [17]:
import zipfile

"""
We extract Tiny ImageNet into DATA_DIR the first time.
Expected structure after unzip:
  data/tiny-imagenet-200/
    train/<wnid>/images/*.JPEG
    val/...
    test/...
Only the 'train' split is used to build our custom dataset.
"""

TINY_ROOT = DATA_DIR / "tiny-imagenet-200"

if not TINY_ROOT.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

print("TINY_ROOT exists:", TINY_ROOT.exists(), "| path:", TINY_ROOT)


TINY_ROOT exists: True | path: C:\Users\veda2\data\tiny-imagenet-200


In [18]:
"""
Fixed numbers from the assignment:
- We want 100 classes, 500 images per class (total 50,000).
- Split per class: Train=300, Val=100, Test=100  (→ 30,000 / 10,000 / 10,000 overall).
- Use a seed for deterministic selection.
"""
import random
random.seed(42)

N_CLASSES   = 100
PER_CLASS   = 500
SPLIT_TRAIN = 300
SPLIT_VAL   = 100
SPLIT_TEST  = 100

assert SPLIT_TRAIN + SPLIT_VAL + SPLIT_TEST == PER_CLASS


In [19]:
"""
We only use Tiny ImageNet's 'train' folders to assemble our custom dataset.
This cell finds all class folders that contain at least 500 images.
"""

train_dir = TINY_ROOT / "train"
all_cls_dirs = sorted([d for d in train_dir.iterdir() if d.is_dir()])

# a class is eligible if it has at least 500 images inside train/<wnid>/images
eligible = []
for d in all_cls_dirs:
    n_imgs = len(list((d / "images").glob("*")))
    if n_imgs >= PER_CLASS:
        eligible.append(d.name)

print(f"Eligible classes: {len(eligible)} (need {N_CLASSES})")
print("Sample eligible:", eligible[:10])


Eligible classes: 200 (need 100)
Sample eligible: ['n01443537', 'n01629819', 'n01641577', 'n01644900', 'n01698640', 'n01742172', 'n01768244', 'n01770393', 'n01774384', 'n01774750']


In [20]:
"""
We pick exactly 100 classes from the eligible set using a fixed seed.
This keeps results reproducible and fair.
"""

if len(eligible) < N_CLASSES:
    raise ValueError(f"Only {len(eligible)} classes have ≥{PER_CLASS} images; need {N_CLASSES}.")

selected = sorted(random.sample(eligible, N_CLASSES))
print("Selected 100 classes (show first 10):", selected[:10])


Selected 100 classes (show first 10): ['n01629819', 'n01698640', 'n01768244', 'n01770393', 'n01774384', 'n01774750', 'n01855672', 'n01950731', 'n02002724', 'n02056570']


In [21]:
"""
Given a list of image paths for one class, this returns:
  - 300 images for train,
  - 100 images for val,
  - 100 images for test.
We shuffle before slicing to avoid ordering bias.
"""

def split_500(files):
    files = files[:]           # copy
    random.shuffle(files)      # shuffle (seed already set globally)
    files = files[:PER_CLASS]  # use exactly 500 for this class
    tr = files[:SPLIT_TRAIN]   # 0 to 300
    va = files[SPLIT_TRAIN:SPLIT_TRAIN+SPLIT_VAL] # 300 to 400
    te = files[SPLIT_TRAIN+SPLIT_VAL:SPLIT_TRAIN+SPLIT_VAL+SPLIT_TEST] # 400 to 500
    return tr, va, te


In [22]:
"""
For each selected class:
  - Read all images in train/<class>/images.
  - Split into (train, val, test) lists using the helper.
We store tuples (class_name, source_path) so we can copy files next.
"""

splits = {"train": [], "val": [], "test": []}

for cls in selected:
    img_paths = sorted((train_dir / cls / "images").glob("*"))
    tr, va, te = split_500(img_paths)
    splits["train"].extend((cls, p) for p in tr)
    splits["val"].extend((cls, p) for p in va)
    splits["test"].extend((cls, p) for p in te)

# quick sanity check: should be 30k/10k/10k
{k: len(v) for k, v in splits.items()}
splits = {"train": [], "val": [], "test": []}

for cls in selected:
    imgs = sorted((TINY_ROOT/"train"/cls/"images").glob("*"))
    tr, va, te = split_500(imgs)
    splits["train"] += [(cls, p) for p in tr]
    splits["val"]   += [(cls, p) for p in va]
    splits["test"]  += [(cls, p) for p in te]

{k: len(v) for k, v in splits.items()}  # expect 30000 / 10000 / 10000


{'train': 30000, 'val': 10000, 'test': 10000}

In [23]:
"""
We now materialize the dataset on disk in a standard ImageFolder layout:

  tiny_100_alexnet/
    train/<class>/*.JPEG     (300 per class)
    val/<class>/*.JPEG       (100 per class)
    test/<class>/*.JPEG      (100 per class)
  train.csv, val.csv, test.csv   # (split, class, relative_path)
  meta.json                      # totals and per-class counts

"""

import csv, json, shutil
from collections import defaultdict

meta = {"splits": {}}

for split in ["train", "val", "test"]:
    split_dir = OUT_DIR / split
    split_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    per_class = defaultdict(int)

    # copy each file into OUT_DIR/<split>/<class>/
    for cls, src in splits[split]:
        dst_dir = split_dir / cls
        dst_dir.mkdir(exist_ok=True)
        dst = dst_dir / src.name
        shutil.copy2(src, dst)  # copy with metadata (portable across OSes)
        rows.append((split, cls, str(dst.relative_to(OUT_DIR)).replace("\\", "/")))
        per_class[cls] += 1

    # write CSV manifest for this split
    with open(OUT_DIR / f"{split}.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["split", "class", "path"])
        w.writerows(rows)

    # add counts to meta
    meta["splits"][split] = {
        "total": len(rows),
        "per_class": dict(sorted(per_class.items()))
    }

# write meta.json summary
with open(OUT_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Totals:", {k: meta["splits"][k]["total"] for k in meta["splits"]})


Totals: {'train': 30000, 'val': 10000, 'test': 10000}


In [27]:
""" We (1) picked 100 distinct classes once, and (2) for each class we shuffled its images,
took the first 500, and then sliced those 500 into non-overlapping chunks:
   0:300  -> train
   300:400 -> val
   400:500 -> test
 Then we copied them into separate folders. Therefore, the same image never appears
 in more than one split (no overlap across train/val/test).
Verifies there are no duplicate file paths across splits in OUT_DIR"""
from pathlib import Path
import csv

OUT_DIR = Path("./tiny_100_alexnet")

def paths_from(csv_name):
    with open(OUT_DIR/csv_name, newline="") as f:
        return {row["path"] for row in csv.DictReader(f)}

train_paths = paths_from("train.csv")
val_paths   = paths_from("val.csv")
test_paths  = paths_from("test.csv")

print("train ∩ val :", len(train_paths & val_paths))
print("train ∩ test:", len(train_paths & test_paths))
print("val   ∩ test:", len(val_paths   & test_paths))


train ∩ val : 0
train ∩ test: 0
val   ∩ test: 0


In [24]:
"""
We confirm:
  - Train has 30,000 images (300 × 100 classes).
  - Val has 10,000 images (100 × 100 classes).
  - Test has 10,000 images (100 × 100 classes).
  - Each split is balanced per class.
"""

import csv
from collections import Counter

def check(split, expected_per_class):
    counts = Counter()
    with open(OUT_DIR / f"{split}.csv", newline="") as f:
        for row in csv.DictReader(f):
            counts[row["class"]] += 1
    total = sum(counts.values())
    balanced = all(v == expected_per_class for v in counts.values())
    return total, balanced

print("TRAIN:", check("train", 300))  # -> (30000, True)
print("VAL  :", check("val",   100))  # -> (10000, True)
print("TEST :", check("test",  100))  # -> (10000, True)


TRAIN: (30000, True)
VAL  : (10000, True)
TEST : (10000, True)


In [25]:
"""
AlexNet preprocessing (Section 2):
  - Resize the shorter side to 256.
  - Train: Random 224×224 crop + random horizontal flip.
  - Val/Test: Center 224×224 crop.
  - Normalize with ImageNet mean/std.
AlexNet steps without torchvision (good for environments where installs fail).
Returns normalized NumPy CHW arrays ready to feed into a model you write later.
"""
from PIL import Image, ImageOps
import numpy as np
import random

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def _resize_shorter(img, size=256):
    w, h = img.size
    if w < h:
        new_w = size; new_h = int(h * (size / w))
    else:
        new_h = size; new_w = int(w * (size / h))
    return img.resize((new_w, new_h), Image.BILINEAR)

def _center_crop(img, size=224):
    return ImageOps.fit(img, (size, size), method=Image.BILINEAR, centering=(0.5, 0.5))

def _random_crop(img, size=224):
    # ensure image is at least 224×224 after resize; then take a random crop
    w, h = img.size
    if w < size or h < size:
        img = ImageOps.fit(img, (max(w, size), max(h, size)), method=Image.BILINEAR)
        w, h = img.size
    x = random.randint(0, w - size)
    y = random.randint(0, h - size)
    return img.crop((x, y, x + size, y + size))

def _maybe_hflip(img, p=0.5):
    return img.transpose(Image.FLIP_LEFT_RIGHT) if random.random() < p else img

def _to_numpy_chw(img):
    # PIL (H, W, C) uint8 [0..255] -> float32 CHW normalized by ImageNet mean/std
    arr = np.asarray(img).astype(np.float32) / 255.0
    if arr.ndim == 2:  # gray → fake RGB
        arr = np.stack([arr, arr, arr], axis=-1)
    chw = np.transpose(arr, (2, 0, 1))
    chw = (chw - IMAGENET_MEAN[:, None, None]) / IMAGENET_STD[:, None, None]
    return chw

def alexnet_train_pil_to_chw(img: Image.Image) -> np.ndarray:
    img = _resize_shorter(img, 256)
    img = _random_crop(img, 224)
    img = _maybe_hflip(img, 0.5)
    return _to_numpy_chw(img)

def alexnet_eval_pil_to_chw(img: Image.Image) -> np.ndarray:
    img = _resize_shorter(img, 256)
    img = _center_crop(img, 224)
    return _to_numpy_chw(img)

print("PIL/Numpy transforms ready (fallback).")


PIL/Numpy transforms ready (fallback).


In [26]:
"""
Dataset: Tiny ImageNet (Stanford CS231n).

Subset rule: 100 classes × 500 images each (50,000 total).

Split: Train 30,000 (300/class), Validation 10,000 (100/class), Test 10,000 (100/class).

Preprocessing (AlexNet §2): Resize shorter side to 256;
Train uses random 224×224 crop + random horizontal flip;
Val/Test use center 224×224 crop;
Normalize with ImageNet mean = (0.485, 0.456, 0.406) and std = (0.229, 0.224, 0.225).

Reproducibility: Fixed seed (=42) for class selection and within-class shuffles. """

'\nDataset: Tiny ImageNet (Stanford CS231n).\n\nSubset rule: 100 classes × 500 images each (50,000 total).\n\nSplit: Train 30,000 (300/class), Validation 10,000 (100/class), Test 10,000 (100/class).\n\nPreprocessing (AlexNet §2): Resize shorter side to 256;\nTrain uses random 224×224 crop + random horizontal flip;\nVal/Test use center 224×224 crop;\nNormalize with ImageNet mean = (0.485, 0.456, 0.406) and std = (0.229, 0.224, 0.225).\n\nReproducibility: Fixed seed (=42) for class selection and within-class shuffles. '